# CatBoost from scratch 
Sections follow the lesson:
1. Why categorical features are hard (label / one-hot / target statistics)
2. Target leakage of naive target statistics (and leave-one-out)
3. Ordered target statistics (random permutation, only "past" rows)
4. Symmetric (oblivious) trees: same split per depth, index arithmetic
5. A mini CatBoost classifier (oblivious trees + Newton leaves + ordered TS)
6. Naive vs ordered TS inside the booster (train/test gap)
7. Ordered boosting and prediction shift (small regression demo)
8. Automatic feature combinations and NaN handling
9. Hyperparameters: depth, l2_leaf_reg, learning rate
10. Optional sanity check against catboost (if installed)

In [ ]:
import os
import time
import warnings

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
T0 = time.time()
os.makedirs("plots", exist_ok=True)
rng = np.random.default_rng(0)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110})


def auc(y, s):
    """ROC-AUC via rank statistic (ties get average rank)."""
    y = np.asarray(y); s = np.asarray(s, float)
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s)); ranks[order] = np.arange(1, len(s) + 1)
    # average ties
    _, inv, cnt = np.unique(s, return_inverse=True, return_counts=True)
    sums = np.bincount(inv, weights=ranks)
    ranks = (sums / cnt)[inv]
    n1 = y.sum(); n0 = len(y) - n1
    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


def logloss(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

## 1. The categorical feature problem
Label encoding invents an order, one-hot explodes for high cardinality (a feature with
500 levels -> 500 columns). Target statistics (TS) replace a category by one number:

`TS(c) = (sum of y in category c + a * prior) / (count in c + a)`

`a` is the smoothing strength, `prior` is usually the global mean of y.
We make a dataset with one **high-cardinality categorical feature that is pure noise**
(it has no relationship with y) plus two honest numeric features. Any model that
looks good by using the category is fooling itself.

In [ ]:
def make_data(n, n_cat=300, seed=0, signal_cats=0):
    r = np.random.default_rng(seed)
    X_num = r.normal(size=(n, 2))
    logit = 1.2 * X_num[:, 0] - 0.8 * X_num[:, 1]
    y = (r.random(n) < sigmoid(logit)).astype(int)
    cat = r.integers(0, n_cat, size=n)             # pure noise "user id"
    return X_num, cat[:, None], y


n_train, n_test, n_cat = 1500, 3000, 300
Xn_tr, Xc_tr, y_tr = make_data(n_train, n_cat, seed=1)
Xn_te, Xc_te, y_te = make_data(n_test, n_cat, seed=2)
print("rows per category (train): mean %.1f" % (n_train / n_cat))
print("one-hot would need", n_cat, "columns for this single feature")

## 2. Naive target statistics leak the label
Naive TS uses *all* rows of the category, including the row being encoded.
For a category with k rows, the row's own label contributes `y_i / (k + a)` to its
own feature: the feature is correlated with the target by construction.
Leave-one-out (LOO) removes the own label, but it leaks in the opposite direction:
`LOO_i = (S - y_i)/(k - 1)` is a *decreasing* function of `y_i` inside a category,
so a tree can recover y_i just from the LOO value.

In [ ]:
def ts_naive(cat, y, a=1.0, prior=None):
    """TS using every row of the category (own label included)."""
    prior = y.mean() if prior is None else prior
    k = np.max(cat) + 1
    s = np.bincount(cat, weights=y, minlength=k)
    c = np.bincount(cat, minlength=k)
    return (s[cat] + a * prior) / (c[cat] + a)


def ts_loo(cat, y, a=1.0, prior=None):
    """Leave-one-out TS: own label removed."""
    prior = y.mean() if prior is None else prior
    k = np.max(cat) + 1
    s = np.bincount(cat, weights=y, minlength=k)
    c = np.bincount(cat, minlength=k)
    return (s[cat] - y + a * prior) / (c[cat] - 1 + a)


def ts_ordered(cat, y, perm, a=1.0, prior=None):
    """Ordered TS: row at permutation position p only sees rows at positions < p.

    Vectorised: sort rows by (category, permutation position); an exclusive cumulative
    sum inside each category group gives the sum/count over the 'past'.
    """
    prior = y.mean() if prior is None else prior
    pos = np.empty(len(perm), int); pos[perm] = np.arange(len(perm))
    order = np.lexsort((pos, cat))               # group by cat, inside group by position
    c_sorted, y_sorted = cat[order], y[order].astype(float)
    cs = np.cumsum(y_sorted) - y_sorted          # exclusive cumsum over everything
    idx = np.arange(len(order))
    start = np.r_[0, np.flatnonzero(np.diff(c_sorted)) + 1]
    group_start = np.repeat(start, np.diff(np.r_[start, len(order)]))
    s_before = cs - cs[group_start]              # sum of y before me in my group
    cnt_before = idx - group_start               # number of rows before me in my group
    out = np.empty(len(order))
    out[order] = (s_before + a * prior) / (cnt_before + a)
    return out


def ts_apply(cat_train, y, cat_new, a=1.0):
    """Statistics for unseen rows: use all training rows (what inference does)."""
    prior = y.mean()
    k = max(cat_train.max(), cat_new.max()) + 1
    s = np.bincount(cat_train, weights=y, minlength=k)
    c = np.bincount(cat_train, minlength=k)
    return (s[cat_new] + a * prior) / (c[cat_new] + a)


c_tr, c_te = Xc_tr[:, 0], Xc_te[:, 0]
perm = rng.permutation(n_train)
enc = {
    "naive": ts_naive(c_tr, y_tr),
    "LOO": ts_loo(c_tr, y_tr),
    "ordered": ts_ordered(c_tr, y_tr, perm),
}
enc_test = ts_apply(c_tr, y_tr, c_te)
print("Single noise feature, AUC of the encoded value alone vs y:")
print(f"{'encoding':10s}{'train AUC':>10s}{'test AUC':>10s}")
for name, e in enc.items():
    print(f"{name:10s}{auc(y_tr, e):10.3f}{auc(y_te, enc_test):10.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, (name, e) in zip(axes[:3], enc.items()):
    ax.scatter(e + rng.normal(0, 0.003, len(e)), y_tr + rng.normal(0, 0.04, len(e)), s=5, alpha=0.4)
    ax.set_title(f"{name}  (train AUC {auc(y_tr, e):.2f})")
    ax.set_xlabel("encoded value")
axes[3].scatter(enc_test + rng.normal(0, 0.003, n_test), y_te + rng.normal(0, 0.04, n_test), s=3, alpha=0.2, c="C3")
axes[3].set_title(f"test, any encoding (AUC {auc(y_te, enc_test):.2f})")
axes[3].set_xlabel("encoded value")
axes[0].set_ylabel("y (jittered)")
plt.tight_layout(); plt.savefig("plots/01_ts_leakage.png"); plt.close()

Naive and LOO encodings look highly predictive **on train** (AUC far above 0.5), yet the
feature is pure noise: on test it is at chance. The ordered encoding stays near 0.5 on
train, i.e. it honestly says "this feature carries no information".

## 3. Ordered target statistics in detail
Worked example from the lesson: rows A,B,C of "Bangladesh" with targets 1,1,0.
With permutation order (A, B, C) and prior 0.5, a = 1:
* A: no history -> 0.5
* B: (1 + 0.5)/(1 + 1) = 0.75
* C: (2 + 0.5)/(2 + 1) = 0.833
Row C never uses its own label.

In [ ]:
cat_toy = np.array([0, 0, 0]); y_toy = np.array([1, 1, 0])
print("naive  :", np.round(ts_naive(cat_toy, y_toy, a=1, prior=0.5), 3))
print("ordered:", np.round(ts_ordered(cat_toy, y_toy, np.array([0, 1, 2]), a=1, prior=0.5), 3))
print("ordered, perm (C,A,B):", np.round(ts_ordered(cat_toy, y_toy, np.array([2, 0, 1]), a=1, prior=0.5), 3))

A single permutation makes early rows noisy (little history). CatBoost therefore uses
several permutations (a different one per tree). Below: variance of the encoding of one
row across permutations, and how the first rows in an ordering have the least information.

In [ ]:
cats_big = rng.integers(0, 10, size=2000)
y_big = (rng.random(2000) < 0.2 + 0.06 * cats_big).astype(float)   # real signal here
true_rate = 0.2 + 0.06 * np.arange(10)
pos_vals, enc_runs = [], []
for _ in range(20):
    p = rng.permutation(2000)
    e = ts_ordered(cats_big, y_big, p)
    enc_runs.append(e)
    pos_vals.append(np.argsort(p))                   # position of each row in this perm
enc_runs = np.array(enc_runs); pos_vals = np.array(pos_vals)
err = np.abs(enc_runs - true_rate[cats_big][None, :]).ravel()
posf = pos_vals.ravel()
bins = np.linspace(0, 2000, 11)
mean_err = [err[(posf >= lo) & (posf < hi)].mean() for lo, hi in zip(bins[:-1], bins[1:])]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].bar(range(10), mean_err, color="C0")
ax[0].set_xticks(range(10)); ax[0].set_xticklabels([f"{int(b)}" for b in bins[:-1]], rotation=45)
ax[0].set_xlabel("position in permutation (start of decile)"); ax[0].set_ylabel("|TS - true rate|")
ax[0].set_title("Early rows have little history")
ax[1].scatter(cats_big[:300] + rng.normal(0, 0.08, 300), enc_runs[0, :300], s=8, label="one perm", alpha=0.6)
ax[1].plot(range(10), true_rate, "k-o", label="true rate")
ax[1].set_xlabel("category"); ax[1].set_ylabel("ordered TS"); ax[1].legend(); ax[1].set_title("Ordered TS tracks the true rate")
plt.tight_layout(); plt.savefig("plots/02_ordered_ts.png"); plt.close()
print("mean |error| first decile %.3f  last decile %.3f" % (mean_err[0], mean_err[-1]))

## 4. Symmetric (oblivious) trees
Every node at the same depth uses the **same** (feature, threshold). A depth-d tree is
then just d splits and an array of 2^d leaves. The leaf index is built with bit
arithmetic: `idx = sum_d (x[f_d] > t_d) << d`, no pointer chasing.

In [ ]:
class ObliviousTree:
    """Symmetric tree fitted on gradient g and hessian h (Newton leaf values)."""

    def __init__(self, depth=4, l2=3.0, n_bins=32):
        self.depth, self.l2, self.n_bins = depth, l2, n_bins

    @staticmethod
    def _go_right(col, thr):
        # NaN rule ("Min" mode): NaN is smaller than everything -> always goes left.
        return np.where(np.isnan(col), False, col > thr)

    def fit(self, X, g, h, rand_strength=0.0, rng=None):
        n, p = X.shape
        self.feats, self.thrs = [], []
        idx = np.zeros(n, dtype=np.int64)
        cand = []
        for j in range(p):
            col = X[:, j]; col = col[~np.isnan(col)]
            qs = np.unique(np.quantile(col, np.linspace(0, 1, self.n_bins + 2)[1:-1])) if len(col) else np.array([])
            cand.append(qs)
        right = [[self._go_right(X[:, j], t) for t in cand[j]] for j in range(p)]
        for d in range(self.depth):
            L = 2 ** d
            best, best_score = None, -np.inf
            for j in range(p):
                for b, t in enumerate(cand[j]):
                    new = idx * 2 + right[j][b]
                    G = np.bincount(new, weights=g, minlength=2 * L)
                    H = np.bincount(new, weights=h, minlength=2 * L)
                    score = np.sum(G ** 2 / (H + self.l2))
                    if rand_strength > 0:
                        score += rand_strength * rng.normal() * np.sqrt(np.abs(score) + 1e-12) * 0.1
                    if score > best_score:
                        best_score, best = score, (j, t, b)
            j, t, b = best
            self.feats.append(j); self.thrs.append(t)
            idx = idx * 2 + right[j][b]
        G = np.bincount(idx, weights=g, minlength=2 ** self.depth)
        H = np.bincount(idx, weights=h, minlength=2 ** self.depth)
        self.leaf = -G / (H + self.l2)               # Newton step with L2 on leaf values
        return self

    def leaf_index(self, X):
        idx = np.zeros(len(X), dtype=np.int64)
        for j, t in zip(self.feats, self.thrs):
            idx = idx * 2 + self._go_right(X[:, j], t)
        return idx

    def predict(self, X):
        return self.leaf[self.leaf_index(X)]


# a 2-D regression example, depth 3 oblivious tree
Xr = rng.uniform(-1, 1, size=(600, 2))
yr = np.sin(3 * Xr[:, 0]) + (Xr[:, 1] > 0.2) * 1.0 + rng.normal(0, 0.1, 600)
tree = ObliviousTree(depth=3, l2=1.0).fit(Xr, g=-(yr - yr.mean()), h=np.ones(600))
print("depth-3 oblivious tree: splits (feature, threshold):",
      [(f, round(float(t), 2)) for f, t in zip(tree.feats, tree.thrs)])
print("number of leaves:", len(tree.leaf), "| numbers stored: 3 splits + 8 leaves (vs 7 splits + 8 leaves)")
gx, gy = np.meshgrid(np.linspace(-1, 1, 200), np.linspace(-1, 1, 200))
grid = np.c_[gx.ravel(), gy.ravel()]
pred = tree.predict(grid).reshape(gx.shape) + yr.mean()
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
cs = ax[0].contourf(gx, gy, pred, levels=12, cmap="viridis")
ax[0].scatter(Xr[:, 0], Xr[:, 1], c=yr, s=6, cmap="viridis", edgecolor="k", linewidth=0.2)
for f, t in zip(tree.feats, tree.thrs):
    (ax[0].axvline if f == 0 else ax[0].axhline)(t, c="w", lw=1.5)
ax[0].set_title("depth-3 oblivious tree: full-width splits")
ax[0].set_xlabel("x0"); ax[0].set_ylabel("x1")
ax[1].bar(range(8), tree.leaf + yr.mean())
ax[1].set_xlabel("leaf index (bits = answers at depth 0,1,2)"); ax[1].set_title("8 leaf values")
plt.tight_layout(); plt.savefig("plots/03_oblivious_tree.png"); plt.close()

Every split line spans the whole plane (unlike an ordinary tree whose second-level splits
only cut one side). That is the built-in regularization: fewer free choices, so
`depth` controls complexity predictably (2^depth leaves).

Speed of "index arithmetic": prediction is `d` vectorised comparisons plus one table lookup.

In [ ]:
Xbig = rng.normal(size=(200000, 2))
t0 = time.time(); _ = tree.predict(Xbig); dt = time.time() - t0
print(f"oblivious tree predicted 200k rows in {dt * 1000:.1f} ms")

## 5. Mini CatBoost classifier
Boosting loop on log loss with gradient `g = p - y`, hessian `h = p(1-p)`:
1. Draw a permutation; encode every categorical column with ordered TS under it.
2. Fit one oblivious tree on (g, h) over numeric + encoded columns.
3. `F += lr * leaf_value`.

At inference, categorical values are encoded with statistics from **all** training rows.
`ts_mode` can be `"ordered"`, `"naive"` or `"loo"` so we can compare them.

In [ ]:
class MiniCatBoost:
    def __init__(self, n_iter=100, lr=0.1, depth=4, l2=3.0, ts_mode="ordered",
                 a=1.0, n_perm=4, seed=0):
        self.n_iter, self.lr, self.depth, self.l2 = n_iter, lr, depth, l2
        self.ts_mode, self.a, self.n_perm, self.seed = ts_mode, a, n_perm, seed

    def _encode_train(self, Xc, y, perm):
        cols = []
        for j in range(Xc.shape[1]):
            if self.ts_mode == "ordered":
                cols.append(ts_ordered(Xc[:, j], y, perm, self.a))
            elif self.ts_mode == "naive":
                cols.append(ts_naive(Xc[:, j], y, self.a))
            elif self.ts_mode == "loo":
                cols.append(ts_loo(Xc[:, j], y, self.a))
        return np.column_stack(cols) if cols else np.zeros((len(y), 0))

    def _encode_new(self, Xc):
        cols = [ts_apply(self.Xc_[:, j], self.y_, Xc[:, j], self.a) for j in range(Xc.shape[1])]
        return np.column_stack(cols) if cols else np.zeros((len(Xc), 0))

    def fit(self, Xn, Xc, y, eval_set=None):
        r = np.random.default_rng(self.seed)
        n = len(y)
        self.Xc_, self.y_ = Xc, y.astype(float)
        self.prior = np.log(y.mean() / (1 - y.mean()))
        F = np.full(n, self.prior)
        perms = [r.permutation(n) for _ in range(self.n_perm)]
        self.trees, self.hist = [], []
        for it in range(self.n_iter):
            enc = self._encode_train(Xc, self.y_, perms[it % self.n_perm])
            X = np.hstack([Xn, enc])
            p = sigmoid(F)
            g, h = p - y, np.maximum(p * (1 - p), 1e-6)
            t = ObliviousTree(self.depth, self.l2).fit(X, g, h)
            F += self.lr * t.predict(X)
            self.trees.append(t)
            if eval_set is not None and (it % 5 == 4 or it == self.n_iter - 1):
                self.hist.append((it + 1, logloss(y, sigmoid(F)),
                                  logloss(eval_set[2], self.predict_proba(eval_set[0], eval_set[1]))))
        return self

    def decision_function(self, Xn, Xc, n_trees=None):
        X = np.hstack([Xn, self._encode_new(Xc)])
        F = np.full(len(X), self.prior)
        for t in self.trees[:n_trees]:
            F += self.lr * t.predict(X)
        return F

    def predict_proba(self, Xn, Xc, n_trees=None):
        return sigmoid(self.decision_function(Xn, Xc, n_trees))


# informative categorical data: 8 categories, true rates differ, plus 1 numeric feature
def make_informative(n, seed):
    r = np.random.default_rng(seed)
    cat = r.integers(0, 8, size=n)
    rate = np.array([0.1, 0.2, 0.3, 0.45, 0.55, 0.7, 0.8, 0.9])
    xn = r.normal(size=(n, 1))
    z = np.log(rate[cat] / (1 - rate[cat])) + 0.8 * xn[:, 0]
    y = (r.random(n) < sigmoid(z)).astype(int)
    return xn, cat[:, None], y


a_n, a_c, a_y = make_informative(2000, 3)
b_n, b_c, b_y = make_informative(4000, 4)
m = MiniCatBoost(n_iter=60, lr=0.15, depth=3, ts_mode="ordered").fit(a_n, a_c, a_y)
print("informative cat feature  train AUC %.3f  test AUC %.3f" % (
    auc(a_y, m.predict_proba(a_n, a_c)), auc(b_y, m.predict_proba(b_n, b_c))))
# Bayes-optimal AUC for reference
rate = np.array([0.1, 0.2, 0.3, 0.45, 0.55, 0.7, 0.8, 0.9])
bz = np.log(rate[b_c[:, 0]] / (1 - rate[b_c[:, 0]])) + 0.8 * b_n[:, 0]
print("oracle test AUC (true logit): %.3f" % auc(b_y, bz))

## 6. Naive vs ordered TS inside the booster
Now the data with the **noise** high-cardinality id (section 1). A model that trusts
the leaked naive encoding will split on it and overfit; the ordered encoding teaches
the trees that the feature is useless.

In [ ]:
res = {}
for mode in ("naive", "loo", "ordered"):
    t0 = time.time()
    mdl = MiniCatBoost(n_iter=60, lr=0.15, depth=4, ts_mode=mode, seed=1).fit(
        Xn_tr, Xc_tr, y_tr, eval_set=(Xn_te, Xc_te, y_te))
    res[mode] = mdl
    print(f"{mode:8s} train AUC {auc(y_tr, mdl.predict_proba(Xn_tr, Xc_tr)):.3f} | "
          f"test AUC {auc(y_te, mdl.predict_proba(Xn_te, Xc_te)):.3f} | "
          f"test logloss {mdl.hist[-1][2]:.3f}  ({time.time() - t0:.1f}s)")
# reference: numeric features only
m_num = MiniCatBoost(n_iter=60, lr=0.15, depth=4, ts_mode="ordered", seed=1).fit(
    Xn_tr, Xc_tr[:, :0], y_tr)
print("numeric only   test AUC %.3f" % auc(y_te, m_num.predict_proba(Xn_te, Xc_te[:, :0])))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for mode, c in zip(("naive", "loo", "ordered"), ("C3", "C1", "C2")):
    h = np.array(res[mode].hist)
    ax[0].plot(h[:, 0], h[:, 1], "--", c=c, label=f"{mode} train")
    ax[0].plot(h[:, 0], h[:, 2], "-", c=c, label=f"{mode} test")
ax[0].set_xlabel("trees"); ax[0].set_ylabel("log loss"); ax[0].legend(fontsize=7, ncol=2)
ax[0].set_title("Noise categorical feature: train/test log loss")
xs = np.arange(3)
tr = [auc(y_tr, res[k].predict_proba(Xn_tr, Xc_tr)) for k in ("naive", "loo", "ordered")]
te = [auc(y_te, res[k].predict_proba(Xn_te, Xc_te)) for k in ("naive", "loo", "ordered")]
ax[1].bar(xs - 0.2, tr, 0.4, label="train"); ax[1].bar(xs + 0.2, te, 0.4, label="test")
ax[1].set_xticks(xs); ax[1].set_xticklabels(["naive", "LOO", "ordered"]); ax[1].set_ylim(0.5, 1)
ax[1].set_ylabel("AUC"); ax[1].legend(); ax[1].set_title("Generalisation gap by encoding")
plt.tight_layout(); plt.savefig("plots/04_booster_naive_vs_ordered.png"); plt.close()

## 7. Ordered boosting and prediction shift
In ordinary boosting the residual of sample i is computed by a model that was trained
**on sample i**, so residuals on train are smaller/biased compared to residuals on fresh
data (prediction shift). Ordered boosting: take a permutation and keep supporting models
`M_k` trained only on the first `2^k` samples; the gradient of sample `i` comes from the
model that has not seen it (`k = floor(log2(i-1))`).

Implementation (regression, squared loss, oblivious trees, a simplified version of the
paper's algorithm): the tree **structure** is chosen from the "unseen-model" gradients; leaf
values of each supporting model use only its prefix; the final model uses all rows.

In [ ]:
class OrderedBoostingRegressor:
    def __init__(self, n_iter=100, lr=0.1, depth=3, l2=1.0, ordered=True, seed=0):
        self.n_iter, self.lr, self.depth, self.l2 = n_iter, lr, depth, l2
        self.ordered, self.seed = ordered, seed

    def fit(self, X, y):
        n = len(y); r = np.random.default_rng(self.seed)
        perm = r.permutation(n); X, y = X[perm], y[perm]       # row i = position i
        self.base = y.mean()
        Ffull = np.full(n, self.base)
        K = int(np.floor(np.log2(n)))                           # prefix sizes 2^0 .. 2^K
        sizes = [2 ** k for k in range(K + 1)]
        Fk = [np.full(n, self.base) for _ in sizes]             # supporting model predictions
        # support index for sample i (0-based): largest k with 2^k <= i (needs i >= 1)
        sup = np.zeros(n, dtype=int)
        for i in range(1, n):
            sup[i] = int(np.floor(np.log2(i)))
        self.trees = []
        for it in range(self.n_iter):
            if self.ordered:
                pred_unseen = np.empty(n)
                for k in range(K + 1):
                    sel = sup == k
                    pred_unseen[sel] = Fk[k][sel]
                pred_unseen[0] = self.base
                g = pred_unseen - y                             # gradient of 0.5*(F-y)^2
            else:
                g = Ffull - y                                   # model has seen every sample
            tree = ObliviousTree(self.depth, self.l2).fit(X, g, np.ones(n))
            leaf = tree.leaf_index(X)
            if self.ordered:
                for k, sz in enumerate(sizes):                  # supporting-model leaf updates
                    G = np.bincount(leaf[:sz], weights=g[:sz], minlength=2 ** self.depth)
                    C = np.bincount(leaf[:sz], minlength=2 ** self.depth)
                    Fk[k] += self.lr * (-G / (C + self.l2))[leaf]
            Ffull += self.lr * tree.leaf[leaf]
            self.trees.append(tree)
        return self

    def predict(self, X, n_trees=None):
        F = np.full(len(X), self.base)
        for t in self.trees[:n_trees]:
            F += self.lr * t.predict(X)
        return F


def make_reg(n, seed, d=6):
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, d))
    y = 2 * X[:, 0] + np.sin(2 * X[:, 1]) + (X[:, 2] > 0) + r.normal(0, 1.5, n)
    return X, y


mse = lambda a, b: float(np.mean((a - b) ** 2))
n_tr = 120
res_ob = {"plain": [], "ordered": []}
curves = {"plain": [], "ordered": []}
for seed in range(6):
    Xa, ya = make_reg(n_tr, 100 + seed); Xb, yb = make_reg(3000, 200 + seed)
    for mode in ("plain", "ordered"):
        mdl = OrderedBoostingRegressor(n_iter=80, lr=0.1, depth=3, l2=1.0,
                                       ordered=(mode == "ordered"), seed=seed).fit(Xa, ya)
        curves[mode].append([(mse(mdl.predict(Xa, k), ya), mse(mdl.predict(Xb, k), yb)) for k in range(1, 81, 4)])
        res_ob[mode].append(curves[mode][-1][-1])
for mode in res_ob:
    a = np.array(res_ob[mode]).mean(0)
    print(f"{mode:8s} train MSE {a[0]:.3f} | test MSE {a[1]:.3f}  (avg of 6 datasets, n={n_tr}, 80 trees)")
print("(noise variance is 2.25; with a tiny n the gap between the two is modest and varies by dataset)")
fig, ax = plt.subplots(figsize=(6.5, 4))
ks = np.arange(1, 81, 4)
for mode, c in (("plain", "C3"), ("ordered", "C2")):
    cu = np.array(curves[mode]).mean(0)
    ax.plot(ks, cu[:, 0], "--", c=c, label=f"{mode} train")
    ax.plot(ks, cu[:, 1], "-", c=c, label=f"{mode} test")
ax.axhline(2.25, c="gray", lw=0.8); ax.set_xlabel("trees"); ax.set_ylabel("MSE")
ax.set_title("Plain vs ordered boosting (small data)"); ax.legend()
plt.tight_layout(); plt.savefig("plots/05_ordered_boosting.png"); plt.close()

Reading: the train curve of plain boosting drops faster (residuals come from a model that
memorised the point), while the ordered version's residuals are honest, so its train
curve tracks test more closely. The price is the extra bookkeeping of `log2(n)`
supporting models, one reason CatBoost is slower than LightGBM.

## 8. Automatic feature combinations and NaN handling
**Combinations.** Target is the XOR of two categorical features: each alone has TS around
0.5 (useless), but the combined categorical feature `A_B` is perfectly informative.
CatBoost builds such combinations greedily; here we do it by hand.

In [ ]:
nx = 3000
A = rng.integers(0, 2, nx); B = rng.integers(0, 2, nx)
yx = ((A ^ B) ^ (rng.random(nx) < 0.05)).astype(int)
AB = A * 2 + B
tr_i, te_i = np.arange(nx) < 2000, np.arange(nx) >= 2000


def fit_eval(Xc):
    mdl = MiniCatBoost(n_iter=40, lr=0.2, depth=2, ts_mode="ordered", seed=0).fit(
        np.zeros((tr_i.sum(), 0)), Xc[tr_i], yx[tr_i])
    return auc(yx[te_i], mdl.predict_proba(np.zeros((te_i.sum(), 0)), Xc[te_i]))


print("A and B separately     test AUC %.3f" % fit_eval(np.c_[A, B]))
print("with combination A_B   test AUC %.3f" % fit_eval(np.c_[A, B, AB]))

**Missing values.** Our oblivious tree sends NaN left (CatBoost "Min" mode: NaN is treated
as smaller than all values). If missingness is informative the tree can isolate it with a
split. Here y depends on whether `x0` is missing.

In [ ]:
nn = 2000
x0 = rng.normal(size=nn); miss = rng.random(nn) < 0.3
yn = (rng.random(nn) < np.where(miss, 0.9, sigmoid(2 * x0))).astype(int)
x0n = np.where(miss, np.nan, x0)
mn = MiniCatBoost(n_iter=30, lr=0.2, depth=2).fit(x0n[:, None], np.zeros((nn, 0), int), yn)
p = mn.predict_proba(x0n[:, None], np.zeros((nn, 0), int))
print("mean predicted P(y=1) when x0 is NaN: %.2f (true 0.90) | when observed: %.2f (true %.2f)" % (
    p[miss].mean(), p[~miss].mean(), yn[~miss].mean()))

## 9. Hyperparameters
Sweep `depth`, `l2_leaf_reg` and `learning_rate` on the informative categorical dataset
plus a noise id column. Symmetric trees make `depth` a clean complexity dial (2^depth leaves).

In [ ]:
def mk(n, seed):
    xn, xc, y = make_informative(n, seed)
    r = np.random.default_rng(seed + 50)
    return xn, np.c_[xc, r.integers(0, 150, size=(n, 1))], y


s_n, s_c, s_y = mk(1200, 7); v_n, v_c, v_y = mk(3000, 8)
sweeps = {"depth": [1, 2, 3, 4, 6], "l2": [0.1, 1, 3, 10, 50], "lr": [0.02, 0.05, 0.1, 0.3, 0.6]}
out = {}
for key, vals in sweeps.items():
    out[key] = []
    for v in vals:
        kw = dict(n_iter=40, lr=0.15, depth=3, l2=3.0)
        kw[{"depth": "depth", "l2": "l2", "lr": "lr"}[key]] = v
        mdl = MiniCatBoost(**kw, seed=2).fit(s_n, s_c, s_y)
        out[key].append((logloss(s_y, mdl.predict_proba(s_n, s_c)), logloss(v_y, mdl.predict_proba(v_n, v_c))))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for a_, (key, vals) in zip(ax, sweeps.items()):
    o = np.array(out[key])
    a_.plot(range(len(vals)), o[:, 0], "o--", label="train"); a_.plot(range(len(vals)), o[:, 1], "o-", label="test")
    a_.set_xticks(range(len(vals))); a_.set_xticklabels(vals); a_.set_xlabel(key); a_.set_ylabel("log loss"); a_.legend()
    print(key, [f"{v}: {t:.3f}/{e:.3f}" for v, (t, e) in zip(vals, out[key])], "(train/test logloss)")
plt.tight_layout(); plt.savefig("plots/06_hyperparameters.png"); plt.close()

## 10. Optional sanity check against the real `catboost`
Not used for any algorithm above; only run if the package happens to be installed.

In [ ]:
try:
    from catboost import CatBoostClassifier
    ref = CatBoostClassifier(iterations=60, learning_rate=0.15, depth=4, verbose=0, random_seed=0)
    ref.fit(np.c_[Xn_tr, Xc_tr], y_tr, cat_features=[2])
    print("catboost test AUC (noise-id data): %.3f" % auc(y_te, ref.predict_proba(np.c_[Xn_te, Xc_te])[:, 1]))
except Exception as e:
    print("catboost not available, skipped:", type(e).__name__)

## Summary
* Naive TS and LOO leak the row's own label -> great train score, chance test score.
* Ordered TS uses only earlier rows of a random permutation -> honest feature.
* Ordered boosting does the same for gradients (supporting models that never saw sample i).
* Oblivious trees: one split per depth, 2^d leaves, bit-index prediction, strong regularizer.
* Combinations, native NaN handling, `l2_leaf_reg`, `depth` are the practical dials.

In [ ]:
print(f"total runtime {time.time() - T0:.1f}s")